In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# -*- coding: utf-8 -*-
"""
#equilibrage_claude.py
Équilibrage automatique du dataset (version simple Colab)

Principe :
- On trouve la classe la plus grande (MAX)
- On augmente les autres jusqu'à MAX
"""

import os
import cv2
import glob
import random
import albumentations as A

# ============================================================
# CHEMIN COLAB (Google Drive)
# ============================================================
OUTPUT_DIR = "/content/drive/MyDrive/dislocation_2026/dataset_classification"

# ============================================================
# AUGMENTATION SIMPLE (réaliste ECCI)
# ============================================================
augment = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.7),
    A.RandomBrightnessContrast(p=0.4),
    A.GaussNoise(p=0.2),
])

# ============================================================
# FONCTION : CHARGER PATCHS
# ============================================================
def load_patches(folder):
    return glob.glob(os.path.join(folder, "*.png"))

# ============================================================
# ÉQUILIBRAGE (MAX AUTO)
# ============================================================
def equilibrer_dataset(classes):

    # 1) compter les classes
    counts = {}
    for c in classes:
        folder = os.path.join(OUTPUT_DIR, c)
        counts[c] = len(load_patches(folder))

    print("\n📊 Avant équilibrage :")
    for c in classes:
        print(f"{c}: {counts[c]}")

    # 2) trouver le MAX
    max_count = max(counts.values())
    print(f"\n🎯 Objectif automatique (MAX) = {max_count}")

    # 3) augmenter chaque classe jusqu'au MAX
    for c in classes:
        folder = os.path.join(OUTPUT_DIR, c)
        patches = load_patches(folder)

        if len(patches) == 0:
            print(f"⚠️ Classe {c} vide")
            continue

        n_to_add = max_count - len(patches)

        print(f"\n📌 Classe {c}")
        print(f"   ajout nécessaire: {n_to_add}")

        for i in range(n_to_add):

            img_path = random.choice(patches)
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

            # augmentation
            augmented = augment(image=img)
            img_aug = augmented["image"]

            # sauvegarde
            name = f"aug_{c}_{i:05d}.png"
            save_path = os.path.join(folder, name)

            cv2.imwrite(save_path, img_aug)

        print(f"   ✅ nouveau total: {len(load_patches(folder))}")

    # 4) résumé final
    print("\n📊 Après équilibrage :")
    for c in classes:
        print(f"{c}: {len(load_patches(os.path.join(OUTPUT_DIR, c)))}")


# ============================================================
# MAIN
# ============================================================
if __name__ == "__main__":

    classes = ["blanc", "noir", "mixte"]

    equilibrer_dataset(classes)

    print("\n✅ Équilibrage terminé (MAX automatique)")


📊 Avant équilibrage :
blanc: 658
noir: 237
mixte: 774

🎯 Objectif automatique (MAX) = 774

📌 Classe blanc
   ajout nécessaire: 116
   ✅ nouveau total: 774

📌 Classe noir
   ajout nécessaire: 537
   ✅ nouveau total: 774

📌 Classe mixte
   ajout nécessaire: 0
   ✅ nouveau total: 774

📊 Après équilibrage :
blanc: 774
noir: 774
mixte: 774

✅ Équilibrage terminé (MAX automatique)
